In [15]:
from copy import deepcopy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)
torch.manual_seed(42)

TARGET_VAR = "ActivePower"
TRAIN_CSV = "train.csv"

USE_HOURLY = False

if USE_HOURLY:
    LOOKBACK = 48
    TRAIN_HORIZON = 72
    FINAL_FORECAST_STEPS = 15 * 24
else:
    LOOKBACK = 144
    TRAIN_HORIZON = 288
    FINAL_FORECAST_STEPS = 15 * 144 

TEST_SIZE = 0.2
BATCH_SIZE = 32 if USE_HOURLY else 16
NUM_EPOCHS = 50
PATIENCE = 10
HIDDEN_SIZE = 32
NUM_LAYERS = 1
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 0.001

In [16]:
df = pd.read_csv(TRAIN_CSV, parse_dates=["Datetime"], index_col="Datetime").sort_index()
df = df.resample("10T").mean()

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET_VAR not in numeric_cols:
    raise ValueError(f"{TARGET_VAR} not in columns")

df_numeric = df[numeric_cols].copy()
for c in df_numeric.columns:
    df_numeric[c] = df_numeric[c].interpolate(method="linear", limit_direction="both")
df_numeric = df_numeric.ffill().bfill()

In [17]:
df.head()

,ActivePower,AmbientTemperatue,BearingShaftTemperature,Blade1PitchAngle,Blade2PitchAngle,Blade3PitchAngle,GearboxBearingTemperature,GearboxOilTemperature,GeneratorRPM,GeneratorWinding1Temperature,GeneratorWinding2Temperature,HubTemperature,MainBoxTemperature,NacellePosition,ReactivePower,RotorRPM,TurbineStatus,WindDirection,WindSpeed
Datetime,,,,,,,,,,,,,,,,,,,
2018-01-01 06:20:00+00:00,26.212347,28.696304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,252.00,3.976499,NaN,NaN,252.00,3.042750
2018-01-01 06:30:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 06:40:00+00:00,59.632658,29.052567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,255.00,11.091660,NaN,NaN,255.00,3.424814
2018-01-01 06:50:00+00:00,40.889650,28.984758,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,264.75,4.234497,NaN,NaN,264.75,3.507172
2018-01-01 07:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
if USE_HOURLY:
    orig = numeric_cols.copy()
    hourly_data = pd.DataFrame(index=df_numeric.resample("H").mean().index)
    hourly_data[TARGET_VAR] = df_numeric[TARGET_VAR].resample("H").sum()
    for col in orig:
        if col == TARGET_VAR:
            continue
        hourly_data[f"{col}_mean"] = df_numeric[col].resample("H").mean()
        hourly_data[f"{col}_std"] = df_numeric[col].resample("H").std().fillna(0)
        hourly_data[f"{col}_min"] = df_numeric[col].resample("H").min()
        hourly_data[f"{col}_max"] = df_numeric[col].resample("H").max()
    df_numeric = hourly_data
    numeric_cols = df_numeric.columns.tolist()

target_idx = numeric_cols.index(TARGET_VAR)
input_size = len(numeric_cols)

In [19]:
df_numeric.head()

,ActivePower,AmbientTemperatue,BearingShaftTemperature,Blade1PitchAngle,Blade2PitchAngle,Blade3PitchAngle,GearboxBearingTemperature,GearboxOilTemperature,GeneratorRPM,GeneratorWinding1Temperature,GeneratorWinding2Temperature,HubTemperature,MainBoxTemperature,NacellePosition,ReactivePower,RotorRPM,TurbineStatus,WindDirection,WindSpeed
Datetime,,,,,,,,,,,,,,,,,,,
2018-01-01 06:20:00+00:00,26.212347,28.696304,47.901936,2.708014,3.192279,3.192279,77.119133,64.204399,1751.715563,113.290871,112.592163,42.996094,41.898426,252.000000,3.976499,15.708135,0.0,252.000000,3.042750
2018-01-01 06:30:00+00:00,42.922502,28.874435,47.901936,2.708014,3.192279,3.192279,77.119133,64.204399,1751.715563,113.290871,112.592163,42.996094,41.898426,253.500000,7.534080,15.708135,0.0,253.500000,3.233782
2018-01-01 06:40:00+00:00,59.632658,29.052567,47.901936,2.708014,3.192279,3.192279,77.119133,64.204399,1751.715563,113.290871,112.592163,42.996094,41.898426,255.000000,11.091660,15.708135,0.0,255.000000,3.424814
2018-01-01 06:50:00+00:00,40.889650,28.984758,47.901936,2.708014,3.192279,3.192279,77.119133,64.204399,1751.715563,113.290871,112.592163,42.996094,41.898426,264.750000,4.234497,15.708135,0.0,264.750000,3.507172
2018-01-01 07:00:00+00:00,40.607409,29.041162,47.901936,2.708014,3.192279,3.192279,77.119133,64.204399,1751.715563,113.290871,112.592163,42.996094,41.898426,265.576087,4.282902,15.708135,0.0,265.576087,3.504348


In [20]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_numeric)


def create_sequences(data, lookback, horizon):
    X, y = [], []
    for i in range(len(data) - lookback - horizon + 1):
        X.append(data[i : i + lookback])
        y.append(data[i + lookback : i + lookback + horizon])
    return np.array(X), np.array(y)


sequences_X, sequences_y = create_sequences(scaled_data, LOOKBACK, TRAIN_HORIZON)
split_idx = int(len(sequences_X) * (1 - TEST_SIZE))
X_train, X_val = sequences_X[:split_idx], sequences_X[split_idx:]
y_train, y_val = sequences_y[:split_idx], sequences_y[split_idx:]

In [21]:
class TS(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return torch.FloatTensor(self.X[i]), torch.FloatTensor(self.y[i])


train_loader = DataLoader(TS(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TS(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)



In [22]:
output_dim = TRAIN_HORIZON * input_size


class GRUModel(nn.Module):
    def __init__(self, in_sz, hidden, out_sz):
        super().__init__()
        self.gru = nn.GRU(in_sz, hidden, NUM_LAYERS, batch_first=True)
        self.fc = nn.Linear(hidden, out_sz)

    def forward(self, x):
        o, _ = self.gru(x)
        return self.fc(o[:, -1, :])


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRUModel(input_size, HIDDEN_SIZE, output_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.7, patience=5)
criterion = nn.MSELoss()

In [23]:
best_val = float("inf")
best_state = None
patience_c = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    tr = 0.0
    for bx, by in train_loader:
        bx = bx.to(device)
        by = by.to(device).reshape(by.shape[0], -1)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr += loss.item()
    tr /= len(train_loader)

    model.eval()
    val_loss = 0.0
    val_sse = 0.0
    val_n = 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx = bx.to(device)
            by = by.to(device).reshape(by.shape[0], -1)
            out = model(bx)
            val_loss += criterion(out, by).item()

            out_np = out.cpu().numpy().reshape(-1, TRAIN_HORIZON, input_size)
            y_np = by.cpu().numpy().reshape(-1, TRAIN_HORIZON, input_size)
            for b in range(out_np.shape[0]):
                pf = np.zeros_like(out_np[b])
                af = np.zeros_like(y_np[b])
                pf[:, target_idx] = out_np[b, :, target_idx]
                af[:, target_idx] = y_np[b, :, target_idx]
                po = scaler.inverse_transform(pf)[:, target_idx]
                ao = scaler.inverse_transform(af)[:, target_idx]
                val_sse += np.sum((po - ao) ** 2)
                val_n += po.size

    val_loss /= len(val_loader)
    val_rmse_original = np.sqrt(val_sse / val_n) if val_n > 0 else np.nan
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        best_state = deepcopy(model.state_dict())
        patience_c = 0
        print(
            f"Epoch {epoch+1}/{NUM_EPOCHS}  train_MSE={tr:.6f}  val_MSE={val_loss:.6f}  "
            f"val_RMSE_{TARGET_VAR}_original={val_rmse_original:.2f}  *"
        )
    else:
        patience_c += 1
        print(
            f"Epoch {epoch+1}/{NUM_EPOCHS}  train_MSE={tr:.6f}  val_MSE={val_loss:.6f}  "
            f"val_RMSE_{TARGET_VAR}_original={val_rmse_original:.2f}"
        )
    if patience_c >= PATIENCE:
        print("Early stopping.")
        break

if best_state is not None:
    model.load_state_dict(best_state)

Epoch 1/50  train_MSE=0.018629  val_MSE=0.018744  val_RMSE_ActivePower_original=415.16  *
Epoch 2/50  train_MSE=0.014614  val_MSE=0.019798  val_RMSE_ActivePower_original=430.46
Epoch 3/50  train_MSE=0.014447  val_MSE=0.019112  val_RMSE_ActivePower_original=414.57
Epoch 4/50  train_MSE=0.014420  val_MSE=0.018955  val_RMSE_ActivePower_original=411.28
Epoch 5/50  train_MSE=0.014451  val_MSE=0.018853  val_RMSE_ActivePower_original=411.55
Epoch 6/50  train_MSE=0.014493  val_MSE=0.019355  val_RMSE_ActivePower_original=409.08
Epoch 7/50  train_MSE=0.014446  val_MSE=0.018501  val_RMSE_ActivePower_original=409.71  *
Epoch 8/50  train_MSE=0.014431  val_MSE=0.018836  val_RMSE_ActivePower_original=413.42
Epoch 9/50  train_MSE=0.014430  val_MSE=0.018930  val_RMSE_ActivePower_original=417.27
Epoch 10/50  train_MSE=0.014426  val_MSE=0.018692  val_RMSE_ActivePower_original=414.17
Epoch 11/50  train_MSE=0.014493  val_MSE=0.018804  val_RMSE_ActivePower_original=417.26
Epoch 12/50  train_MSE=0.014412  va

In [24]:
def predict_recursive(model, last_sequence, total_steps, device):
    model.eval()
    current = np.array(last_sequence, copy=True)
    chunks = []
    remaining = total_steps
    with torch.no_grad():
        while remaining > 0:
            pred = model(torch.FloatTensor(current).unsqueeze(0).to(device))
            pred_np = pred.cpu().numpy().reshape(TRAIN_HORIZON, input_size)
            take = min(TRAIN_HORIZON, remaining)
            chunks.append(pred_np[:take])
            remaining -= take
            if remaining <= 0:
                break
            current = np.vstack([current[take:], pred_np[:take]])
    return np.vstack(chunks)


last_seq = scaled_data[-LOOKBACK:]
final_scaled = predict_recursive(model, last_seq, FINAL_FORECAST_STEPS, device)
final_original = scaler.inverse_transform(final_scaled)

last_ts = df_numeric.index[-1]
if USE_HOURLY:
    idx = pd.date_range(last_ts + pd.Timedelta(hours=1), periods=FINAL_FORECAST_STEPS, freq="h")
else:
    idx = pd.date_range(last_ts + pd.Timedelta(minutes=10), periods=FINAL_FORECAST_STEPS, freq="10T")

pred_df = pd.DataFrame(final_original, columns=numeric_cols, index=idx)
pred_df.index.name = "Datetime"
pred_df.to_csv("predictions_next_15_days.csv", index=True)

In [25]:
if TARGET_VAR in pred_df.columns:
    daily_totals = pred_df[TARGET_VAR].resample("D").sum()
    daily_totals.name = "predicted_total_power_generated"
    daily_totals.to_csv(
        "predictions_next_15_days_daily_power_totals.csv",
        header=["predicted_total_power_generated"],
    )
    print("predictions_next_15_days_daily_power_totals.csv")
    print(daily_totals)

torch.save(model.state_dict(), "gru_forecast_model.pt")
print("gru_forecast_model.pt saved.")

predictions_next_15_days_daily_power_totals.csv
Datetime
2020-03-16 00:00:00+00:00    83836.992188
2020-03-17 00:00:00+00:00    80378.796875
2020-03-18 00:00:00+00:00    59274.863281
2020-03-19 00:00:00+00:00    57367.675781
2020-03-20 00:00:00+00:00    56380.125000
2020-03-21 00:00:00+00:00    54643.718750
2020-03-22 00:00:00+00:00    56069.687500
2020-03-23 00:00:00+00:00    54351.871094
2020-03-24 00:00:00+00:00    56034.261719
2020-03-25 00:00:00+00:00    54318.640625
2020-03-26 00:00:00+00:00    56029.898438
2020-03-27 00:00:00+00:00    54314.558594
2020-03-28 00:00:00+00:00    56029.308594
2020-03-29 00:00:00+00:00    54314.007812
2020-03-30 00:00:00+00:00    56029.222656
Freq: D, Name: predicted_total_power_generated, dtype: float32
gru_forecast_model.pt saved.


In [26]:
s = pred_df[TARGET_VAR].values
if USE_HOURLY:
    assert len(s) >= 15 * 24
    daily_15 = s.reshape(15, 24).sum(axis=1)
else:
    assert len(s) >= 15 * 144
    daily_15 = s.reshape(15, 144).sum(axis=1)
pd.Series(daily_15, index=[f"day_{i+1}" for i in range(15)]).to_csv(
    "predictions_15_days_one_row_per_day.csv", header=["predicted_total_power_that_day"]
)